<a href="https://colab.research.google.com/github/DAS-Bodhi/Fed-Recc/blob/main/Baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import random
from sklearn.model_selection import train_test_split

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

PyTorch version: 2.11.0+cu128
GPU available: True


**A. Pytorch-** The core deep learning library.Gives Tensors(GPU-accelerated multi-dimensional arrays) and Autograd (for GD, so that no need to derive by hand).

**B. torch.nn-** PyTorch's sub-module for building neural network layers and models. gives us -
1. nn.Module- The base class every model inherits from. gets you automatic parameter tracking, device-moving, state-dict save/load (used constantly in FedAvg loop to send weights between server and clients)
2. nn.Embedding- lookup table mapping an integer ID to a learnable vector.
3. nn.Parameter — marks a raw tensor as something the optimizer should track and update.
4. nn.MSELoss — Mean Squared Error loss, used to compare predicted ratings to actual ratings during training.

**C. numpy-** The standard library for numerical array operations in Python.used in code as ***np.array_split(all_user_indices, N_CLIENTS)*** in Cell 6 — this splits shuffled list of 943 user indices into 10 roughly-equal chunks (some clients get 94 users, some 95, since 943 doesn't divide evenly by 10)

**D. pandas-** The standard library for tabular data (spreadsheet-like tables) in Python. Used in loading data, re-indexing and partitioning data based on clients.

**E. random-** Python's built-in pseudo-random number module.for seeding which helps in setting a starting point for the "random" shuffle, so it's not actually random every time — it's the same shuffle every time you run the celland shuffling data between clients.

**F. train_test_split-** It takes your full DataFrame (100,000 rating rows) and randomly divides it into two separate DataFrames for training and testing with no overlap between them. Every row ends up in exactly one of the two.

**G. CUDA-** It is the software layer that lets PyTorch actually talk to an NVIDIA GPU and hand it that work. Without CUDA, PyTorch could only ever use the CPU, no matter what hardware is physically in the machine.

In [ ]:
!wget -q https://files.grouplens.org/datasets/movielens/ml-100k.zip
!unzip -oq ml-100k.zip

cols = ["user_id", "item_id", "rating", "timestamp"]
df = pd.read_csv("ml-100k/u.data", sep="\t", names=cols)
print(df.shape)
df.head()

(100000, 4)


,user_id,item_id,rating,timestamp
0,196,242,3,881250949
1,186,302,3,891717742
2,22,377,1,878887116
3,244,51,2,880606923
4,166,346,1,886397596


Downloads the MovieLens 100K zip from GroupLens, unzips it, and loads just the u.data file (the core ratings: user ID, item ID, rating, timestamp) into a pandas DataFrame — giving it column names since the raw file has none. Ends with a shape check (should be 100,000 rows × 4 columns) and a preview of the first 5 rows, to confirm the download and load worked correctly before building anything on top of it.


**Justification for using only u.data** -This dissertation's core model is Matrix Factorisation, which predicts a rating purely from a user ID, an item ID, and the target rating — it has no mechanism to consume genre, title, age, occupation, or any other attribute. u.data contains exactly the three fields the model needs (plus timestamp, which isn't used either).

In [ ]:
user_ids = df['user_id'].unique()
item_ids = df['item_id'].unique()

n_users = len(user_ids)
n_items = len(item_ids)
print(f"n_users={n_users}, n_items={n_items}")

user2idx = {}
for i, u in enumerate(user_ids):
    user2idx[u] = i

item2idx = {}
for i, it in enumerate(item_ids):
    item2idx[it] = i

df['user_idx'] = df['user_id'].map(user2idx)
df['item_idx'] = df['item_id'].map(item2idx)

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

df[['user_id', 'user_idx', 'item_id', 'item_idx']].head(10)

n_users=943, n_items=1682


,user_id,user_idx,item_id,item_idx
0,196,0,242,0
1,186,1,302,1
2,22,2,377,2
3,244,3,51,3
4,166,4,346,4
5,298,5,474,5
6,115,6,265,6
7,253,7,465,7
8,305,8,451,8
9,6,9,86,9


**PREPROCESSING**


1. **Get unique IDs —** pull out every distinct user (943) and item (1682) that appears in the data, with duplicates removed. Stored as NumPy arrays, though they behave like lists for our purposes.
2. **Count them —** n_users, n_items store those counts, needed later to size the model's embedding tables (one row per user, one row per item).
3. **Build the translation dictionaries —** user2idx and item2idx map each raw ID to a clean index, assigned in the order the ID first appears in the data (not sorted) and create two references.
4. **Apply the translation —** Goes through every row of the full 100,000-row DataFrame and looks up each raw ID against the dictionary, writing the result into two new columns: user_idx, item_idx.
5. **Split into train/test —** randomly divides the (now re-indexed) data into 80,000 training rows and 20,000 test rows. random_state=42 fixes the randomness so the split is identical every time the notebook re-runs for reproducible results.
6. **Visual check —** placed as the last line of the cell, auto-displays the first 10 rows showing raw ID next to new index side by side, so the mapping can be eyeballed. No display() needed since Jupyter/Colab auto-renders whatever the final line of a cell returns.



In [ ]:
class MatrixFactorization(nn.Module):
    def __init__(self, n_users, n_items, emb_dim=64):
      #structural specs
        super().__init__()
        #tracking begins for the params (embeddings and biases)
        self.user_emb = nn.Embedding(n_users, emb_dim)
        self.item_emb = nn.Embedding(n_items, emb_dim)
        self.user_bias = nn.Embedding(n_users, 1)
        self.item_bias = nn.Embedding(n_items, 1)
        self.global_bias = nn.Parameter(torch.zeros(1))

        nn.init.normal_(self.user_emb.weight, std=0.01)
        nn.init.normal_(self.item_emb.weight, std=0.01)
        nn.init.zeros_(self.user_bias.weight)
        nn.init.zeros_(self.item_bias.weight)

    def forward(self, user, item):
        dot = (self.user_emb(user) * self.item_emb(item)).sum(dim=1)
        return (dot + self.user_bias(user).squeeze()
                     + self.item_bias(item).squeeze()
                     + self.global_bias)

1. user_emb — [943, 64], each user's taste vector
2. item_emb — [1682, 64], each item's vector
3. user_bias — [943, 1], each user's rating tendency
4. item_bias — [1682, 1], each item's rating tendency
5. global_bias — [1], one shared average (nn.Parameter, not nn.Embedding, since it's a single value, not a lookup table)

Initialization — embeddings get small random values to break symmetry between the 64 dimensions; biases get exact zero, safe since each is a single scalar with nothing to differentiate.

Forward - the prediction function, called automatically. Takes a *batch* of user/item indices (not single values) and computes all predictions in that batch at once:

$$\hat{r}_{u,i} = b_0 + b_u + b_i + \mathbf{p}_u \cdot \mathbf{q}_i$$
	​
.squeeze() — strips an unwanted trailing dimension. self.user_bias(user) returns shape [batch_size, 1]; dot is shape [batch_size]. Without squeezing, adding them would broadcast into an unintended [batch_size, batch_size] grid instead of adding element-by-element.


In [ ]:
# --- Helper function: measures model performance on any given dataset split ---
def evaluate(model, df_split):
    model.eval()  # switch to eval mode (no functional effect here, but standard practice)

    # convert the given DataFrame's columns into tensors on the GPU/CPU
    u = torch.tensor(df_split['user_idx'].values, dtype=torch.long, device=device)
    i = torch.tensor(df_split['item_idx'].values, dtype=torch.long, device=device)
    r = torch.tensor(df_split['rating'].values, dtype=torch.float32, device=device)

    with torch.no_grad():  # don't track gradients — we're only measuring, not training
        preds = model(u, i)  # calls forward() from Cell 4 — one prediction per row
        # compute MSE, take its square root to get RMSE, then get the raw value
        rmse = torch.sqrt(nn.functional.mse_loss(preds, r)).item()

    return rmse  # hand back the plain float RMSE value


# --- Create the actual model instance (first real object built from the Cell 4 blueprint) ---
model = MatrixFactorization(n_users, n_items, emb_dim=64).to(device)  # .to(device) moves all params to GPU/CPU

# --- Set up the training tools ---
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=1e-5)
# model.parameters() hands the optimizer every learnable piece (works because of
# super().__init__() back in Cell 4, which registered them)

loss_fn = nn.MSELoss()  # reusable MSE loss object, called every epoch below


# --- Convert the training data into tensors ONCE, outside the loop (reused every epoch) ---
u_train = torch.tensor(train_df['user_idx'].values, dtype=torch.long, device=device)
i_train = torch.tensor(train_df['item_idx'].values, dtype=torch.long, device=device)
r_train = torch.tensor(train_df['rating'].values, dtype=torch.float32, device=device)


# --- Early stopping tracking variables, initialized before training starts ---
best_rmse = float('inf')   # starts impossibly high, so epoch 0 always counts as "best so far"
best_state = None          # will hold a snapshot of the model's best weights
patience = 23               # how many non-improving epochs to tolerate before giving up
bad_epochs = 0               # running counter of consecutive non-improving epochs


# --- Main training loop ---
for epoch in range(300):  # epoch takes values 0, 1, 2, ... up to 299 (or until early stop)

    model.train()             # switch to training mode
    optimizer.zero_grad()     # clear gradients left over from the previous epoch

    preds = model(u_train, i_train)   # forward pass — all 80,000 ratings predicted at once
    loss = loss_fn(preds, r_train)     # compute this epoch's MSE loss

    loss.backward()    # autograd computes gradients for all 168,626 parameters
    optimizer.step()    # Adam uses those gradients to actually update every parameter

    # --- check performance on the held-out test set, AFTER this epoch's update ---
    test_rmse = evaluate(model, test_df)   # calls the function defined at the top

    if test_rmse < best_rmse:
        # new best found — save the RMSE, snapshot the current weights, reset the bad-streak counter
        best_rmse, best_state, bad_epochs = test_rmse, model.state_dict(), 0
    else:
        # no improvement this epoch
        bad_epochs += 1
        if bad_epochs >= patience:
            # 15 non-improving epochs in a row — stop training early
            print(f"Early stopping at epoch {epoch}, best RMSE={best_rmse:.4f}")
            break   # exits the for loop immediately, skipping remaining epochs

    # --- progress logging every 10th epoch (cosmetic only, doesn't affect training) ---
    if epoch % 10 == 0:
        print(f"Epoch {epoch}: test_RMSE={test_rmse:.4f}")


# --- after the loop ends: restore the BEST weights ever seen, not necessarily the last ones ---
model.load_state_dict(best_state)

print(f"\nCentralized benchmark RMSE: {best_rmse:.4f}")

Epoch 0: test_RMSE=3.6705
Epoch 10: test_RMSE=3.0158
Epoch 20: test_RMSE=1.3815
Epoch 30: test_RMSE=1.2816
Epoch 40: test_RMSE=0.9857
Epoch 50: test_RMSE=0.9961
Epoch 60: test_RMSE=0.9461
Epoch 70: test_RMSE=0.9343
Epoch 80: test_RMSE=0.9361
Epoch 90: test_RMSE=0.9326
Epoch 100: test_RMSE=0.9344
Early stopping at epoch 110, best RMSE=0.9323

Centralized benchmark RMSE: 0.9323


Prediction, r-hat:
$$\hat{r}_n = b_0 + b_{u_n} + b_{i_n} + \mathbf{p}_{u_n} \cdot \mathbf{q}_{i_n}$$
Computed by forward() in Cell 4, called via preds = model(u_train, i_train)
in Cell 5. One prediction per training example, all 80000 computed at once.

r_n (the real rating):
Comes directly from train_df['rating'], converted to r_train — never
computed, just read from the data.

Loss (MSE), computed inside the epoch loop in Cell 5:
$$\mathcal{L} = \frac{1}{N}\sum_{n=1}^{N} (\hat{r}_n - r_n)^2, \qquad N = 80000$$
Called as: loss = loss_fn(preds, r_train)

RMSE — computed only inside evaluate(), never inside the training loop itself:
$$\text{RMSE} = \sqrt{\mathcal{L}} = \sqrt{\frac{1}{N}\sum_{n=1}^{N} (\hat{r}_n - r_n)^2}$$
Called as: test_rmse = evaluate(model, test_df)
Note: the training loop computes MSE (loss) to train on; RMSE is only
computed separately, on test_df, purely to monitor and compare performance —
it never feeds back into training itself.

theta — every learnable parameter across all 5 pieces of the model:
user_emb, item_emb, user_bias, item_bias, global_bias (168626 numbers total).

Gradient, computed by loss.backward():
$$g_t = \frac{\partial \mathcal{L}}{\partial \theta}$$
This is a DIRECTION, not a step. A positive g_t means "increasing this
theta would increase loss" -> so theta should move down (decrease).
A negative g_t means "increasing this theta would decrease loss" -> so
theta should move up (increase). The MAGNITUDE of g_t signals how
sharply loss responds to that parameter right now — a larger |g_t| means
a small change in theta swings loss a lot; a smaller |g_t| means loss is
fairly flat with respect to that parameter at this point.

How the direction is actually used, in optimizer.step():
$$\theta_{t+1} = \theta_t - \eta\, \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon}$$

$$m_t,\ v_t \; \text{are both running averages derived from } g_t: \; m_t \text{ smooths its direction, } v_t \text{ smooths its magnitude squared.}$$

The minus sign is what converts "direction of increasing loss" into
"step toward decreasing loss" — moving opposite to the raw gradient's
direction, scaled by Adam's smoothed momentum and adaptive step size.

Early stopping condition, checked every epoch:
$$\text{if } \; \text{RMSE}_{\text{test}}^{(t)} \geq \min_{k \le t} \big(\text{RMSE}_{\text{test}}^{(k)}\big) \; \text{for 15 consecutive epochs} \rightarrow \text{stop}$$

Final restored weights:
$$\theta^{*} = \theta_{t^*}, \qquad t^* = \arg\min_t \; \text{RMSE}_{\text{test}}^{(t)}$$
Restored via: model.load_state_dict(best_state)

In [ ]:
N_CLIENTS = 10
all_user_indices = list(range(n_users))
random.seed(42)
random.shuffle(all_user_indices)

client_user_splits = np.array_split(all_user_indices, N_CLIENTS)

client_data = []
for client_id, users_in_client in enumerate(client_user_splits):
    users_in_client_set = set(users_in_client)
    client_train_df = train_df[train_df['user_idx'].isin(users_in_client_set)]
    client_data.append(client_train_df)
    print(f"Client {client_id}: {len(users_in_client)} users, {len(client_train_df)} ratings")

Client 0: 95 users, 6985 ratings
Client 1: 95 users, 7042 ratings
Client 2: 95 users, 8732 ratings
Client 3: 94 users, 8547 ratings
Client 4: 94 users, 8952 ratings
Client 5: 94 users, 9083 ratings
Client 6: 94 users, 8482 ratings
Client 7: 94 users, 7228 ratings
Client 8: 94 users, 6894 ratings
Client 9: 94 users, 8055 ratings


Two-step process: first split who belongs to which client (just index bookkeeping, no rating data yet), then fetch each client's actual ratings.

A. Split the indices:

all_user_indices — list of all 943 user indices (0 to 942)
Shuffled (fixed seed 42) so client groups are random cross-sections, not ordered blocks
np.array_split(..., 10) — divides into 10 chunks (~94-95 users each, since 943 doesn't divide evenly)

B. Load each client's ratings:

For each chunk of user indices, filter train_df down to only rows whose user_idx is in that chunk
.isin(users_in_client) does the actual filtering — one True/False check per row of train_df
Store each client's filtered DataFrame into client_data (a list of 10 DataFrames)

End result: client_data[0] through client_data[9] — each one a private slice of train_df, containing only ratings from that client's ~94 assigned users. No client can see another's data — this is the structural foundation that makes the FedAvg simulation meaningful.

Note on partition type: this is an IID partition — random, statistically similar across clients. Real-world federated data is typically non-IID (each client's data skewed toward their own behavior). Worth naming as a scope simplification in the Methodology/Limitations chapter.

In [ ]:
# --- Simulates one client's local training, for ONE client, called once per client per round ---
def local_train(global_state_dict, client_df, n_users, n_items, emb_dim=64, local_epochs=5, lr=0.01):

    # create a BRAND NEW model object — a separate copy, not shared with other clients or the global model
    local_model = MatrixFactorization(n_users, n_items, emb_dim).to(device)

    # overwrite this fresh model's random init weights with the CURRENT global weights —
    # this is how every round starts from wherever the shared model currently is
    local_model.load_state_dict(global_state_dict)

    # separate optimizer, created fresh every call — so no momentum (m_t, v_t) carries
    # over between different clients, or between different rounds for the same client
    local_optimizer = torch.optim.Adam(local_model.parameters(), lr=lr, weight_decay=1e-5)
    local_loss_fn = nn.MSELoss()

    # convert THIS CLIENT's data only (client_df) into tensors — not the full 80000-row train_df
    users_t = torch.tensor(client_df['user_idx'].values, dtype=torch.long, device=device)
    items_t = torch.tensor(client_df['item_idx'].values, dtype=torch.long, device=device)
    ratings_t = torch.tensor(client_df['rating'].values, dtype=torch.float32, device=device)

    local_model.train()
    for epoch in range(local_epochs):   # only 5 epochs — short local training, not 300
        local_optimizer.zero_grad()               # clear gradients from the previous local epoch
        preds = local_model(users_t, items_t)      # forward pass on THIS client's data only
        loss = local_loss_fn(preds, ratings_t)      # loss computed only against this client's ratings
        loss.backward()                              # gradients computed only from this client's data
        local_optimizer.step()                        # update this client's LOCAL copy of the weights

        # note: no evaluate() call here, no early stopping — local training just runs
        # its fixed 5 epochs and stops, unlike the centralized loop in Cell 5

    # return the updated LOCAL weights (never the raw ratings), plus how many ratings
    # this client had — used later by the server to weight this client's contribution
    return local_model.state_dict(), len(client_df)

Cell 7 — Local client training

local\_train() simulates one client's on-device training. Called once
per client, every round (10 times per round, 300 rounds total).

Step-by-step:

1. Fresh, isolated model created each call:
$$\theta_{\text{local}} \leftarrow \text{MatrixFactorization}(n_{\text{users}}, n_{\text{items}}, \text{emb\_dim})$$

2. Global weights loaded in — local training always starts from the
current shared state:
$$\theta_{\text{local}} \leftarrow \theta_{\text{global}}^{(t)}$$

3. Local training loop, for $E=5$ epochs, using ONLY this client's own
data $(u_c, i_c, r_c)$:
$$\mathcal{L}_c = \frac{1}{N_c}\sum_{n=1}^{N_c} (\hat{r}_n - r_n)^2$$
$$\theta_{\text{local}} \leftarrow \theta_{\text{local}} - \eta\, \frac{\hat{m}_t}{\sqrt{\hat{v}_t}+\epsilon}, \quad \text{repeated } E \text{ times}$$

4. Returned to the server:
$$(\theta_{\text{local}}^{\text{updated}},\; N_c)$$
where $N_c = |\text{client\_df}|$ — the number of ratings this client
had, used later to WEIGHT this client's contribution in the federated
average. Raw ratings never leave this function — only $\theta$ and $N_c$.

Relevance: the gap between what the server ever sees ($\theta_{\text{local}}^{\text{updated}}$)
and what actually produced it (the client's private $r_c$) is exactly
the privacy boundary the dissertation's attacks target.

In [ ]:
def federated_average(client_states, client_sizes):
    total_size = sum(client_sizes)
    avg_state = {}

    for key in client_states[0].keys():
        weighted_sum = 0
        for i in range(len(client_states)):
            client_weight = client_sizes[i] / total_size
            weighted_sum += client_states[i][key].float() * client_weight
        avg_state[key] = weighted_sum

    return avg_state

FedAvg aggregation (server-side)

federated_average() combines 10 clients' locally-trained weights into
one new global model, weighted by how much data each client had.

For each parameter theta (user_emb, item_emb, user_bias, item_bias,
global_bias), computed independently:

Each client's share of the total data:
$$w_c = \frac{N_c}{N}, \qquad N = \sum_{c=1}^{10} N_c$$

The weighted average across all 10 clients, for this one parameter:
$$\theta_{\text{global}}^{(t+1)} = \sum_{c=1}^{10} w_c \, \theta_c$$

where theta_c is client c's locally-trained value for this parameter
(from local_train() in Cell 7), and N_c = len(client_df) for that client.

Structure of the code:
- Outer loop: iterates over WHICH PARAMETER (5 total: user_emb.weight,
  item_emb.weight, user_bias.weight, item_bias.weight, global_bias)
- Inner loop: iterates over WHICH CLIENT (10 total), accumulating each
  client's weighted contribution for the parameter the outer loop is
  currently on

Key property: this is a WEIGHTED average, not a plain average — a
client with more ratings (larger N_c) pulls the result more toward its
own values; a client with fewer ratings contributes proportionally less.
This is exactly the "Averaging" in Federated Averaging (McMahan et al.).

Result: avg_state, a complete state_dict() ready to be loaded into the
global model via global_model.load_state_dict(avg_state) in Cell 9.

In [ ]:
global_model = MatrixFactorization(n_users, n_items, emb_dim=64).to(device)
NUM_ROUNDS = 300

for rnd in range(NUM_ROUNDS):
    global_state = global_model.state_dict()
    client_states, client_sizes = [], []

    for client_df in client_data:
        state, n_samples = local_train(global_state, client_df, n_users, n_items)
        client_states.append(state)
        client_sizes.append(n_samples)

    new_global_state = federated_average(client_states, client_sizes)
    global_model.load_state_dict(new_global_state)

    if rnd % 5 == 0 or rnd == NUM_ROUNDS - 1:
        rmse = evaluate(global_model, test_df)
        print(f"Round {rnd}: test_RMSE={rmse:.4f}")

final_fedavg_rmse = evaluate(global_model, test_df)
print(f"\nFinal FedAvg RMSE: {final_fedavg_rmse:.4f}")

Round 0: test_RMSE=3.5992
Round 5: test_RMSE=3.0482
Round 10: test_RMSE=2.5204
Round 15: test_RMSE=2.0495
Round 20: test_RMSE=1.6952
Round 25: test_RMSE=1.4649
Round 30: test_RMSE=1.3167
Round 35: test_RMSE=1.2167
Round 40: test_RMSE=1.1464
Round 45: test_RMSE=1.0961
Round 50: test_RMSE=1.0604
Round 55: test_RMSE=1.0358
Round 60: test_RMSE=1.0193
Round 65: test_RMSE=1.0091
Round 70: test_RMSE=1.0029
Round 75: test_RMSE=1.0001
Round 80: test_RMSE=0.9987
Round 85: test_RMSE=0.9977
Round 90: test_RMSE=0.9969
Round 95: test_RMSE=0.9963
Round 100: test_RMSE=0.9957
Round 105: test_RMSE=0.9952
Round 110: test_RMSE=0.9948
Round 115: test_RMSE=0.9944
Round 120: test_RMSE=0.9940
Round 125: test_RMSE=0.9937
Round 130: test_RMSE=0.9935
Round 135: test_RMSE=0.9932
Round 140: test_RMSE=0.9930
Round 145: test_RMSE=0.9928
Round 150: test_RMSE=0.9926
Round 155: test_RMSE=0.9924
Round 160: test_RMSE=0.9923
Round 165: test_RMSE=0.9921
Round 170: test_RMSE=0.9920
Round 175: test_RMSE=0.9919
Round 180: tes